# Convert .h5 → INT8 TFLite (+ Edge TPU)

Self-contained notebook for converting any Dean-architecture model trained with `train_ablation.py`. Handles Drive mount, py310 venv with TF 2.15.0, INT8 PTQ with 200-image calibration, static-shape wrap (Edge-TPU-safe), Edge TPU compilation, smoke test, and **automatic output-mapping detection** (TFLite often reorders the angle/speed outputs).

## How to use

1. **Run Cell 1 (SETUP) once per Colab session.** Mounts Drive, builds the venv, installs `edgetpu_compiler`. ~5 min first time, instant on later runs in the same runtime.
2. **Edit `EXP_NAME` (and `CROP_TOP` if needed) in Cell 2 (CONVERT) and run** for each model.
3. **Optional: use Cell 3 (BATCH) to convert multiple models in one pass.**

## Files needed on Drive root (`My Drive/`)

- `picar_data.zip` — training data zip
- `requirements.txt` — pinned versions
- `picar_models/<EXP_NAME>_best.h5` — the trained model to convert

## Outputs

- `picar_models_tflite/<EXP_NAME>_int8.tflite` — CPU INT8 TFLite
- `picar_models_tflite/<EXP_NAME>_int8_edgetpu.tflite` — Edge-TPU-compiled

Drop both into your Pi folder under `src/picar_autopilot_models/<name>_tpu/` (rename to `model_int8.tflite` and `model_edgetpu.tflite`).

## Cell 1 — SETUP (run once per runtime)

In [ ]:
import os, shutil

# ─── 1. Drive mount (idempotent + force-remount if stub state) ──────────────
from google.colab import drive
if not os.path.ismount('/content/drive'):
    if os.path.exists('/content/drive'):
        try: drive.flush_and_unmount()
        except Exception: pass
        if not os.path.ismount('/content/drive'):
            shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive', force_remount=True)
else:
    print('⏭️  Drive already mounted')

# ─── 2. Verify required files on Drive ──────────────────────────────────────
ZIP_PATH = '/content/drive/MyDrive/picar_data.zip'
REQ_PATH = '/content/drive/MyDrive/requirements.txt'
for p in [ZIP_PATH, REQ_PATH]:
    assert os.path.exists(p), f'❌ missing on Drive: {p}'
print('✅ Drive files OK')

# ─── 3. Unzip training data (skip if already done) ──────────────────────────
if not os.path.exists('/content/PiCar/data/training_data/training_data'):
    !mkdir -p /content/PiCar
    !cd /content/PiCar && unzip -q -o {ZIP_PATH}
    print('✅ Unzipped data')
else:
    print('⏭️  Data already unzipped')

os.makedirs('/content/PiCar', exist_ok=True)
shutil.copy(REQ_PATH, '/content/PiCar/requirements.txt')

# ─── 4. Build py310 venv with TF 2.15.0 + CUDA wheels ───────────────────────
VENV     = '/content/py310'
VENV_PY  = f'{VENV}/bin/python'
VENV_PIP = f'{VENV}/bin/pip'

if not os.path.exists(VENV_PY):
    # Disable PPAs that intermittently time out and break apt
    !sudo rm -f /etc/apt/sources.list.d/ubuntugis*.list
    !sudo rm -f /etc/apt/sources.list.d/deadsnakes*.list
    !sudo rm -f /etc/apt/sources.list.d/graphics-drivers*.list
    !sudo apt-get update -q
    !sudo apt-get install -y --fix-missing python3.10-venv python3.10-dev python3-setuptools-whl python3-pip-whl -q
    !python3.10 -m venv {VENV}
    assert os.path.exists(VENV_PY), 'venv shell missing — apt install of python3.10-venv likely failed'
    !{VENV_PIP} install -q --upgrade pip
    !{VENV_PIP} install -q "numpy>=1.23.5,<2.0"
    !{VENV_PIP} install -q -r /content/PiCar/requirements.txt
    # CUDA wheels — tensorflow[and-cuda]==2.15.0 would have installed these but
    # its tensorrt-libs==8.6.1 is no longer on PyPI. Install individually.
    !{VENV_PIP} install -q \
        "nvidia-cublas-cu12==12.2.5.6" \
        "nvidia-cuda-cupti-cu12==12.2.142" \
        "nvidia-cuda-nvcc-cu12==12.2.140" \
        "nvidia-cuda-nvrtc-cu12==12.2.140" \
        "nvidia-cuda-runtime-cu12==12.2.140" \
        "nvidia-cudnn-cu12==8.9.4.25" \
        "nvidia-cufft-cu12==11.0.8.103" \
        "nvidia-curand-cu12==10.3.3.141" \
        "nvidia-cusolver-cu12==11.5.2.141" \
        "nvidia-cusparse-cu12==12.1.2.141" \
        "nvidia-nccl-cu12==2.16.5" \
        "nvidia-nvjitlink-cu12==12.2.140"
    print('✅ py310 venv built')
else:
    print('⏭️  venv already exists')

# Verify TF 2.15.0
import subprocess
v = subprocess.run([VENV_PY, '-c',
    "import tensorflow as tf, keras; print(tf.__version__, keras.__version__)"
], capture_output=True, text=True)
print(f'venv: {v.stdout.strip()}   (expect: 2.15.0 2.15.0)')
assert '2.15.0' in v.stdout, 'venv TF/Keras not 2.15.0'

# ─── 5. Install Edge TPU compiler ──────────────────────────────────────────
if not os.path.exists('/usr/bin/edgetpu_compiler'):
    !curl -s https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add - 2>/dev/null
    !echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | \
         sudo tee /etc/apt/sources.list.d/coral-edgetpu.list >/dev/null
    !sudo apt-get update -q
    !sudo apt-get install -y edgetpu-compiler -q
    print('✅ edgetpu_compiler installed')
else:
    print('⏭️  edgetpu_compiler already installed')

OUT_DIR = '/content/drive/MyDrive/picar_models_tflite'
os.makedirs(OUT_DIR, exist_ok=True)
print('\n══════════ SETUP DONE ══════════')

## Cell 2 — CONVERT a single model

Edit `EXP_NAME` and `CROP_TOP` (default 110, set 120 for `crop120_30` ablations), then run.

The cell will:
1. Build a 200-image calibration set with the right crop
2. Wrap the model with static input shape (`batch_size=1`)
3. Convert to INT8 TFLite
4. Sanity-check (assert int8 dtype, non-zero scale, static shape)
5. **Detect output ordering** (TFLite often reorders angle/speed; we parse `StatefulPartitionedCall:N` to recover the truth)
6. Smoke-test 5 different images (both heads should vary)
7. Compile for Edge TPU

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  EDIT THESE                                                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
EXP_NAME  = '50_kaggle_16_clean'   # change for each model
CROP_TOP  = 110                     # 110 default; set 120 for crop120_30 ablations
# ────────────────────────────────────────────────────────────────────────────

import os
CROP_BOTTOM, H, W, C = 30, 96, 160, 3
OUT_DIR    = '/content/drive/MyDrive/picar_models_tflite'
H5_PATH    = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
TFLITE_OUT = f'{OUT_DIR}/{EXP_NAME}_int8.tflite'
TPU_OUT    = f'{OUT_DIR}/{EXP_NAME}_int8_edgetpu.tflite'
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(H5_PATH), f'❌ {H5_PATH} not on Drive'

# ─── Write conversion script (runs in the venv with Keras 2.15) ────────────
CONV_SCRIPT = '/content/_convert.py'
with open(CONV_SCRIPT, 'w') as f:
    f.write(f'''
import os, numpy as np, pandas as pd, tensorflow as tf

EXP_NAME, H5_PATH, TFLITE_OUT = "{EXP_NAME}", "{H5_PATH}", "{TFLITE_OUT}"
CROP_TOP, CROP_BOTTOM, H, W, C = {CROP_TOP}, {CROP_BOTTOM}, {H}, {W}, {C}
print(f"crop=[{{CROP_TOP}}:-{{CROP_BOTTOM}}], resize={{H}}x{{W}}x{{C}}")
print(f"Loading {{H5_PATH}}")

# Build calibration set (200 random training images preprocessed like training)
df = pd.read_csv("/content/PiCar/data/train_clean_weighted.csv")
sample_ids = df.sample(200, random_state=42)["image_id"].astype(int).astype(str).tolist()
img_dir = "/content/PiCar/data/training_data/training_data"

calib = np.empty((200, H, W, C), dtype=np.float32)
n = 0
for fid in sample_ids:
    p = os.path.join(img_dir, f"{{fid}}.png")
    if not os.path.exists(p): continue
    img = tf.io.read_file(p)
    img = tf.image.decode_png(img, channels=C)
    img = tf.cast(img, tf.float32) / 255.0
    img = img[CROP_TOP:-CROP_BOTTOM, :, :]
    img = tf.image.resize(img, [H, W])
    calib[n] = img.numpy(); n += 1
calib = calib[:n]
print(f"calib: {{calib.shape}}, range [{{calib.min():.3f}}, {{calib.max():.3f}}]")
assert calib.min() >= -0.05 and calib.max() <= 1.05, "calibration not in [0, 1]"

# Load + force static input shape (Edge-TPU-safe wrap)
base = tf.keras.models.load_model(H5_PATH, compile=False)
inputs  = tf.keras.Input(shape=(H, W, C), batch_size=1)
outputs = base(inputs, training=False)
model   = tf.keras.Model(inputs, outputs)
_ = model(np.zeros((1, H, W, C), dtype=np.float32))
print(f"Model wrapped with static shape (1, {{H}}, {{W}}, {{C}})")

# INT8 conversion
def representative_dataset():
    for img in calib:
        yield [np.expand_dims(img, axis=0)]

c = tf.lite.TFLiteConverter.from_keras_model(model)
c.optimizations = [tf.lite.Optimize.DEFAULT]
c.representative_dataset = representative_dataset
c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
c.inference_input_type  = tf.int8
c.inference_output_type = tf.int8

print("Converting to INT8 TFLite...")
tflite_bytes = c.convert()
with open(TFLITE_OUT, "wb") as f: f.write(tflite_bytes)
print(f"Saved {{TFLITE_OUT}} ({{len(tflite_bytes)/1024:.1f}} KB)")

# Sanity check
interp = tf.lite.Interpreter(model_path=TFLITE_OUT)
interp.allocate_tensors()
in_d   = interp.get_input_details()[0]
out_ds = interp.get_output_details()
print(f"\\nInput: shape={{list(in_d['shape'])}} dtype={{in_d['dtype'].__name__}} "
      f"q=(scale={{in_d['quantization'][0]:.6f}}, zp={{in_d['quantization'][1]}})")
for i, od in enumerate(out_ds):
    print(f"Output {{i}}: name={{od.get('name','?')!r}} shape={{list(od['shape'])}} "
          f"q=(scale={{od['quantization'][0]:.6f}}, zp={{od['quantization'][1]}})")

assert in_d["dtype"].__name__ == "int8" and in_d["quantization"][0] != 0, "PTQ failed"
assert tuple(in_d["shape"]) == (1, H, W, C), "input shape not static"
print("INT8 fully applied + static shape confirmed")

# Detect output mapping. Keras model defines outputs=[angle_out, speed_out] so:
#   - StatefulPartitionedCall:0 → angle (Keras output 0)
#   - StatefulPartitionedCall:1 → speed (Keras output 1)
# But TFLite often enumerates them in REVERSE order. Use the name suffix to recover.
angle_idx, speed_idx = None, None
for i, od in enumerate(out_ds):
    n = od.get("name", "")
    nl = n.lower()
    if "angle" in nl: angle_idx = i
    elif "speed" in nl: speed_idx = i
    elif "StatefulPartitionedCall" in n:
        suffix = n.rsplit(":", 1)[-1]
        if suffix == "0" and angle_idx is None: angle_idx = i
        elif suffix == "1" and speed_idx is None: speed_idx = i
if angle_idx is None: angle_idx = 0
if speed_idx is None: speed_idx = 1
swap_label = "⚠️  SWAPPED — out0=speed, out1=angle" if angle_idx == 1 else "normal — out0=angle, out1=speed"
print(f"\\n→ Detected output mapping: angle_idx={{angle_idx}}, speed_idx={{speed_idx}}  ({{swap_label}})")

# Smoke test — both heads should vary across 5 different calibration images
print("\\nSmoke test (BOTH heads should vary):")
in_scale, in_zp = in_d["quantization"]
for k in [0, 50, 100, 150, min(195, len(calib)-1)]:
    x = (calib[k] / in_scale + in_zp).round().astype(np.int8)
    interp.set_tensor(in_d["index"], np.expand_dims(x, 0))
    interp.invoke()
    a_raw = int(interp.get_tensor(out_ds[angle_idx]["index"]).flatten()[0])
    s_raw = int(interp.get_tensor(out_ds[speed_idx]["index"]).flatten()[0])
    a_s, a_z = out_ds[angle_idx]["quantization"]
    s_s, s_z = out_ds[speed_idx]["quantization"]
    a_f = (a_raw - a_z) * a_s
    s_f = (s_raw - s_z) * s_s
    print(f"  img[{{k:3d}}]: angle int8={{a_raw:5d}} -> {{a_f:.4f}}  |  speed int8={{s_raw:5d}} -> {{s_f:.4f}}")
''')

# ─── Run conversion via the py310 venv ─────────────────────────────────────
print(f'\n══════════ CONVERTING {EXP_NAME} (CROP_TOP={CROP_TOP}) ══════════')
!/content/py310/bin/python {CONV_SCRIPT}

# ─── Compile for Edge TPU ─────────────────────────────────────────────────
print(f'\n══════════ EDGE TPU COMPILE {EXP_NAME} ══════════')
!edgetpu_compiler -o {OUT_DIR} {TFLITE_OUT}

# ─── Summary ──────────────────────────────────────────────────────────────
print(f'\n══════════ DONE: {EXP_NAME} ══════════')
print(f'INT8 CPU:  {TFLITE_OUT}')
if os.path.exists(TPU_OUT):
    print(f'EdgeTPU:   {TPU_OUT}')
else:
    print('⚠️  Edge TPU compile did not produce output — check log above')

## Cell 3 — BATCH CONVERT (optional)

Convert + compile multiple models in one pass. Edit the `MODELS` list (each entry is `(exp_name, crop_top)`).

In [ ]:
# (exp_name, crop_top) — one entry per model to convert
MODELS = [
    ('60_baseline16_clean_rerun', 110),
    ('61_huber_angle_only_rerun', 110),
    ('62_crop120_30_rerun',       120),  # different crop
    ('63_progressive_rerun',      110),
    ('64_cutout_smaller_rerun',   110),
    ('65_cutout_aggressive_rerun',110),
    ('66_dense_smaller_rerun',    110),
    ('67_bn_locked_rerun',        110),
]

import os
OUT_DIR = '/content/drive/MyDrive/picar_models_tflite'
os.makedirs(OUT_DIR, exist_ok=True)

results = []
for exp, ct in MODELS:
    h5 = f'/content/drive/MyDrive/picar_models/{exp}_best.h5'
    if not os.path.exists(h5):
        print(f'⏭️  {exp}: not on Drive, skipping')
        results.append((exp, False, 'no .h5 on Drive'))
        continue
    print(f'\n═══════════════════ {exp} (CROP_TOP={ct}) ═══════════════════')

    script = '/content/_convert_batch.py'
    with open(script, 'w') as f:
        f.write(f'''
import os, numpy as np, pandas as pd, tensorflow as tf

EXP_NAME = "{exp}"
H5_PATH = "{h5}"
TFLITE_OUT = "{OUT_DIR}/{exp}_int8.tflite"
CROP_TOP, CROP_BOTTOM, H, W, C = {ct}, 30, 96, 160, 3

df = pd.read_csv("/content/PiCar/data/train_clean_weighted.csv")
sample_ids = df.sample(200, random_state=42)["image_id"].astype(int).astype(str).tolist()
img_dir = "/content/PiCar/data/training_data/training_data"
calib = np.empty((200, H, W, C), dtype=np.float32)
n = 0
for fid in sample_ids:
    p = os.path.join(img_dir, f"{{fid}}.png")
    if not os.path.exists(p): continue
    img = tf.io.read_file(p)
    img = tf.image.decode_png(img, channels=C)
    img = tf.cast(img, tf.float32) / 255.0
    img = img[CROP_TOP:-CROP_BOTTOM, :, :]
    img = tf.image.resize(img, [H, W])
    calib[n] = img.numpy(); n += 1
calib = calib[:n]

base = tf.keras.models.load_model(H5_PATH, compile=False)
inp = tf.keras.Input(shape=(H, W, C), batch_size=1)
out = base(inp, training=False)
model = tf.keras.Model(inp, out)
_ = model(np.zeros((1, H, W, C), dtype=np.float32))

def repset():
    for img in calib: yield [np.expand_dims(img, axis=0)]

c = tf.lite.TFLiteConverter.from_keras_model(model)
c.optimizations = [tf.lite.Optimize.DEFAULT]
c.representative_dataset = repset
c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
c.inference_input_type = tf.int8
c.inference_output_type = tf.int8
b = c.convert()
with open(TFLITE_OUT, "wb") as f: f.write(b)
print(f"saved INT8: {{len(b)/1024:.1f}} KB")

# Quick smoke test
interp = tf.lite.Interpreter(model_path=TFLITE_OUT)
interp.allocate_tensors()
in_d, out_ds = interp.get_input_details()[0], interp.get_output_details()
in_s, in_z = in_d["quantization"]
vals = [[], []]
for k in [0, 50, 100, 150, min(195, len(calib)-1)]:
    x = (calib[k] / in_s + in_z).round().astype(np.int8)
    interp.set_tensor(in_d["index"], np.expand_dims(x, 0))
    interp.invoke()
    for i in (0, 1):
        s, z = out_ds[i]["quantization"]
        v = (int(interp.get_tensor(out_ds[i]["index"]).flatten()[0]) - z) * s
        vals[i].append(v)
v0_var = max(vals[0]) - min(vals[0])
v1_var = max(vals[1]) - min(vals[1])
print(f"out0 range=[{{min(vals[0]):.3f}}, {{max(vals[0]):.3f}}] var={{v0_var:.4f}}")
print(f"out1 range=[{{min(vals[1]):.3f}}, {{max(vals[1]):.3f}}] var={{v1_var:.4f}}")
if v0_var < 0.01 and v1_var < 0.01:
    print("⚠️  BOTH outputs are constant — model has dead head(s)")
elif v0_var < 0.01 or v1_var < 0.01:
    dead = "out0" if v0_var < 0.01 else "out1"
    print(f"⚠️  {{dead}} is constant — dead head, but the other one is healthy")
else:
    print("✅ Both outputs vary")
''')
    !/content/py310/bin/python {script}
    !edgetpu_compiler -o {OUT_DIR} {OUT_DIR}/{exp}_int8.tflite 2>&1 | tail -3
    edgetpu_path = f'{OUT_DIR}/{exp}_int8_edgetpu.tflite'
    results.append((exp, os.path.exists(edgetpu_path), 'ok' if os.path.exists(edgetpu_path) else 'edgetpu compile failed'))

print('\n═══════════════════ SUMMARY ═══════════════════')
for exp, ok, msg in results:
    icon = '✅' if ok else '❌'
    print(f'{icon}  {exp:35s}  {msg}')

## After conversion — deploy to Pi

On your Mac, for each converted model:

```bash
# 1. Make a Pi folder and drop the TPU template + experiment_details.json
EXP=62_crop120_30_rerun
mkdir -p src/picar_autopilot_models/${EXP}_tpu
cp src/picar_autopilot_models/_template_tpu/model.py \
   src/picar_autopilot_models/${EXP}_tpu/model.py
# Create experiment_details.json with crop info (or copy from training output)

# 2. Download .tflite files from Drive and rename:
cp ~/Downloads/${EXP}_int8.tflite          src/picar_autopilot_models/${EXP}_tpu/model_int8.tflite
cp ~/Downloads/${EXP}_int8_edgetpu.tflite  src/picar_autopilot_models/${EXP}_tpu/model_edgetpu.tflite

# 3. SCP to Pi and test
scp -r src/picar_autopilot_models/${EXP}_tpu pi@192.168.50.1:~/autopilot/autopilot/models/
ssh pi@192.168.50.1 'cd ~/autopilot && python3 run.py --model '${EXP}_tpu' --mode drive --duration 60'
```

The TPU template `model.py` already auto-detects the angle/speed output ordering via `StatefulPartitionedCall:N`, so it works regardless of which way TFLite chose to enumerate the outputs.